# Exposure -> Conversion Join and Attribution Window Comparison

## Beginner-friendly summary
This notebook uses a real-world event dataset to build explicit exposure->conversion joins, then deepens attribution analysis with additional rule families, segment sensitivity, and robustness checks.

### What this notebook does
- Uses a real-world attribution dataset (Criteo Attribution Modeling Dataset)
- Builds an exposure->conversion joined table with clear lookback rules
- Compares attribution methods (last-touch, linear, time-decay, U-shape, W-shape, Markov/Shapley-style proxies)
- Compares attribution outputs across windows (1, 3, 7, 14, 30 days) and lag filters
- Measures segment-level sensitivity (user touch-count cohort, dominant channel cohort, campaign cohort)
- Produces an attribution-ready measurement table for downstream reporting/modeling

### Major steps (in order)
1. Download/load real dataset
2. Prepare exposure and conversion event tables
3. Build windowed exposure->conversion joins
4. Attribution method comparison
5. Window and lag robustness comparison
6. Segment-level sensitivity analysis
7. Final measurement-ready output

### Side notes for beginners
- Join window defines how far back an exposure can receive conversion credit.
- Attribution method choice can change channel credit materially.
- Segment sensitivity helps us see whether a method is stable across user/channel groups.
- This notebook focuses on measurement logic, not churn/CTR model training.


## 1) Setup

> Side note: We keep dependencies lightweight (`pandas`, `numpy`) so the notebook is easy to run.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

## 2) Download Real-World Dataset (Criteo Attribution)

Dataset source:
- [Criteo Attribution Modeling Dataset](https://huggingface.co/datasets/criteo/criteo-attribution-dataset)

Direct file used in this notebook:
- `criteo_attribution_dataset.tsv.gz`

> Side note: The raw file is large. We read a configurable row sample for fast iteration.

In [ ]:
import urllib.request

data_dir = Path('data')
data_dir.mkdir(parents=True, exist_ok=True)

dataset_path = data_dir / 'criteo_attribution_dataset.tsv.gz'
dataset_url = 'https://huggingface.co/datasets/criteo/criteo-attribution-dataset/resolve/main/criteo_attribution_dataset.tsv.gz'

if not dataset_path.exists():
    print('Downloading real-world dataset (this can take time)...')
    urllib.request.urlretrieve(dataset_url, dataset_path)
    print('Download complete:', dataset_path)
else:
    print('Dataset already present:', dataset_path)

## 3) Load and Prepare Event Tables

Expected raw columns (tab-separated, no header):
- `timestamp`
- `uid`
- `campaign`
- `conversion`
- `conversion_timestamp`

> Side note: A single user may have many touchpoints and multiple conversions.

In [ ]:
TARGET_CONVERTED_USERS = 2000
CHUNK_SIZE = 500_000
cols = ['timestamp', 'uid', 'campaign', 'conversion', 'conversion_timestamp', 'conversion_id']

# Pass 1: collect users who have at least one conversion event
converted_users = set()
for chunk in pd.read_csv(dataset_path, sep='\t', compression='gzip', usecols=cols, chunksize=CHUNK_SIZE, low_memory=False):
    chunk['conversion'] = pd.to_numeric(chunk['conversion'], errors='coerce').fillna(0).astype(int)
    conv = chunk[chunk['conversion'] == 1]
    if not conv.empty:
        converted_users.update(conv['uid'].dropna().astype(str).tolist())
    if len(converted_users) >= TARGET_CONVERTED_USERS:
        break

selected_users = set(list(converted_users)[:TARGET_CONVERTED_USERS])
print('Selected converted users:', len(selected_users))

# Pass 2: collect all touchpoints for selected users
parts = []
for chunk in pd.read_csv(dataset_path, sep='\t', compression='gzip', usecols=cols, chunksize=CHUNK_SIZE, low_memory=False):
    chunk['uid'] = chunk['uid'].astype(str)
    keep = chunk[chunk['uid'].isin(selected_users)]
    if not keep.empty:
        parts.append(keep)

if not parts:
    raise RuntimeError('No rows found for selected converted users. Try increasing TARGET_CONVERTED_USERS.')

raw = pd.concat(parts, ignore_index=True)

raw['timestamp'] = pd.to_numeric(raw['timestamp'], errors='coerce')
raw['conversion_timestamp'] = pd.to_numeric(raw['conversion_timestamp'], errors='coerce')
raw['conversion'] = pd.to_numeric(raw['conversion'], errors='coerce').fillna(0).astype(int)
raw['conversion_id'] = pd.to_numeric(raw['conversion_id'], errors='coerce')
raw = raw.dropna(subset=['timestamp', 'uid', 'campaign']).copy()
raw['ts_exposure'] = pd.to_datetime(raw['timestamp'], unit='s', errors='coerce')

# Exposure table
exposures = raw[['uid', 'campaign', 'timestamp', 'ts_exposure']].rename(columns={'uid': 'user_id', 'campaign': 'channel'})
exposures['exposure_id'] = np.arange(len(exposures), dtype=np.int64)

# Conversion table: one row per distinct conversion event
conv_rows = raw[(raw['conversion'] == 1) & (raw['conversion_timestamp'].notna())].copy()
conv_rows = conv_rows[conv_rows['conversion_timestamp'] >= conv_rows['timestamp']]

conversions = conv_rows[['uid', 'conversion_id', 'conversion_timestamp']].rename(columns={'uid': 'user_id'}).drop_duplicates().reset_index(drop=True)
conversions['ts_conversion'] = pd.to_datetime(conversions['conversion_timestamp'], unit='s', errors='coerce')
conversions = conversions.dropna(subset=['ts_conversion']).reset_index(drop=True)
conversions['conversion_id'] = np.arange(len(conversions), dtype=np.int64)
conversions['conversion_value'] = 1.0

print('Filtered raw rows:', len(raw))
print('Exposure rows:', len(exposures))
print('Conversion rows:', len(conversions))
print('Unique users in filtered set:', exposures['user_id'].nunique())

display(exposures.head(5))
display(conversions.head(5))


## 4) Exposure -> Conversion Join (Windowed)

Join rule:
- same `user_id`
- exposure occurred before conversion
- exposure within lookback window (in days)

> Side note: This is the auditable core of measurement pipelines.

In [ ]:
def build_join_table(exposures_df: pd.DataFrame, conversions_df: pd.DataFrame, lookback_days: int) -> pd.DataFrame:
    merged = conversions_df.merge(exposures_df, on='user_id', how='left')
    delta_hours = (merged['ts_conversion'] - merged['ts_exposure']).dt.total_seconds() / 3600

    valid = merged[(delta_hours >= 0) & (delta_hours <= lookback_days * 24)].copy()
    if valid.empty:
        return pd.DataFrame(columns=[
            'lookback_days', 'conversion_id', 'user_id', 'ts_conversion', 'conversion_value',
            'exposure_id', 'channel', 'ts_exposure', 'hours_before_conversion'
        ])

    valid['hours_before_conversion'] = delta_hours.loc[valid.index]
    valid['lookback_days'] = lookback_days

    cols = [
        'lookback_days', 'conversion_id', 'user_id', 'ts_conversion', 'conversion_value',
        'exposure_id', 'channel', 'ts_exposure', 'hours_before_conversion'
    ]
    return valid[cols].sort_values(['conversion_id', 'ts_exposure']).reset_index(drop=True)

joined_7d = build_join_table(exposures, conversions, lookback_days=7)
print('Joined rows (7-day):', len(joined_7d))
print('Unique conversions covered (7-day):', joined_7d['conversion_id'].nunique())
print('Coverage % (7-day):', round(100 * joined_7d['conversion_id'].nunique() / max(len(conversions), 1), 2))
display(joined_7d.head(10))


## 4b) Coverage Summary (Explicit)

> Side note: Coverage tells us what fraction of conversions can be linked to at least one prior exposure in the selected lookback window.


In [ ]:
total_conversions = int(conversions['conversion_id'].nunique())
matched_conversions_7d = int(joined_7d['conversion_id'].nunique())
coverage_pct_7d = round(100 * matched_conversions_7d / max(total_conversions, 1), 2)

coverage_summary = pd.DataFrame([{
    'lookback_days': 7,
    'total_conversions': total_conversions,
    'matched_conversions': matched_conversions_7d,
    'coverage_pct': coverage_pct_7d
}])

print(f"Coverage formula: matched_conversions / total_conversions = {matched_conversions_7d} / {total_conversions} = {coverage_pct_7d}%")
display(coverage_summary)


## 5) Attribution Method Comparison (Expanded)

Methods included:
- Last-touch
- Linear multi-touch
- Time-decay
- U-shape (position-based)
- W-shape (position-based)
- Markov-style removal proxy
- Shapley-style proxy

> Side note: Markov/Shapley here are practical proxy implementations for educational measurement workflow.


In [ ]:
def _normalize_weights(s: pd.Series) -> pd.Series:
    total = float(s.sum())
    if total <= 0:
        n = len(s)
        return pd.Series(np.ones(n) / max(n, 1), index=s.index)
    return s / total

def _position_rank(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(['conversion_id', 'ts_exposure']).copy()
    out['pos'] = out.groupby('conversion_id').cumcount()
    out['n'] = out.groupby('conversion_id')['exposure_id'].transform('count')
    return out

def attrib_last_touch(join_df: pd.DataFrame) -> pd.DataFrame:
    idx = join_df.groupby('conversion_id')['ts_exposure'].idxmax()
    out = join_df.loc[idx, ['channel', 'conversion_value']].copy()
    out['credit'] = out['conversion_value']
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_linear(join_df: pd.DataFrame) -> pd.DataFrame:
    cnt = join_df.groupby('conversion_id')['exposure_id'].transform('count')
    out = join_df[['channel', 'conversion_value']].copy()
    out['credit'] = out['conversion_value'] / cnt
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_time_decay(join_df: pd.DataFrame, half_life_hours: float = 48.0) -> pd.DataFrame:
    out = join_df[['conversion_id', 'channel', 'conversion_value', 'hours_before_conversion']].copy()
    out['w'] = np.exp(-np.log(2) * out['hours_before_conversion'] / half_life_hours)
    out['w'] = out.groupby('conversion_id')['w'].transform(lambda x: _normalize_weights(x))
    out['credit'] = out['conversion_value'] * out['w']
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_u_shape(join_df: pd.DataFrame) -> pd.DataFrame:
    out = _position_rank(join_df)
    pos = out['pos']
    n = out['n']
    w = np.where(
        n == 1,
        1.0,
        np.where(
            n == 2,
            0.5,
            np.where(pos == 0, 0.4, np.where(pos == (n - 1), 0.4, 0.2 / (n - 2)))
        )
    )
    out['w'] = w
    out['credit'] = out['conversion_value'] * out['w']
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_w_shape(join_df: pd.DataFrame) -> pd.DataFrame:
    out = _position_rank(join_df)
    pos = out['pos']
    n = out['n']
    mid = (n // 2)
    anchor = (pos == 0) | (pos == (n - 1)) | (pos == mid)
    anchor_count = anchor.groupby(out['conversion_id']).transform('sum').astype(float)
    rem_count = (n - anchor_count).astype(float)

    w_main = np.where(anchor, 0.9 / np.maximum(anchor_count, 1.0), np.where(rem_count > 0, 0.1 / rem_count, 0.0))
    w_small = 1.0 / np.maximum(n, 1)
    w = np.where(n <= 2, w_small, w_main)

    out['w'] = w
    out['credit'] = out['conversion_value'] * out['w']
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_markov_proxy(join_df: pd.DataFrame) -> pd.DataFrame:
    # Proxy: channel removal effect approximated by conversion-path participation share.
    paths = join_df.groupby('conversion_id')['channel'].apply(lambda x: tuple(pd.unique(x)))
    channels = sorted(join_df['channel'].dropna().unique())
    scores = []
    for ch in channels:
        affected = paths.apply(lambda p: ch in p).sum()
        scores.append((ch, float(affected)))
    out = pd.DataFrame(scores, columns=['channel', 'score'])
    out['score'] = _normalize_weights(out['score'])
    total_val = float(join_df.drop_duplicates('conversion_id')['conversion_value'].sum())
    out['credit'] = out['score'] * total_val
    return out[['channel', 'credit']]

def attrib_shapley_proxy(join_df: pd.DataFrame) -> pd.DataFrame:
    # Proxy: per-conversion unique-channel sharing (Shapley-style equal marginal under symmetric coalition assumption).
    temp = join_df.groupby('conversion_id').agg(
        conversion_value=('conversion_value', 'first'),
        channels=('channel', lambda x: list(pd.unique(x)))
    ).reset_index()
    rows = []
    for _, r in temp.iterrows():
        chs = [c for c in r['channels'] if pd.notna(c)]
        if not chs:
            continue
        share = float(r['conversion_value']) / len(chs)
        for ch in chs:
            rows.append((ch, share))
    out = pd.DataFrame(rows, columns=['channel', 'credit'])
    return out.groupby('channel', as_index=False)['credit'].sum()

ATTRIBUTION_METHODS = {
    'last_touch': attrib_last_touch,
    'linear': attrib_linear,
    'time_decay': attrib_time_decay,
    'u_shape': attrib_u_shape,
    'w_shape': attrib_w_shape,
    'markov_proxy': attrib_markov_proxy,
    'shapley_proxy': attrib_shapley_proxy,
}

def run_method(join_df: pd.DataFrame, method_name: str) -> pd.DataFrame:
    f = ATTRIBUTION_METHODS[method_name]
    out = f(join_df).copy()
    out['method'] = method_name
    return out[['method', 'channel', 'credit']]

method_table_7d = pd.concat([run_method(joined_7d, m) for m in ATTRIBUTION_METHODS], ignore_index=True)
method_pivot_7d = method_table_7d.pivot_table(index='channel', columns='method', values='credit', aggfunc='sum').fillna(0)
display(method_pivot_7d)



## 6) Robustness Checks: Windows and Lag Filters

We compare methods over:
- windows: 1, 3, 7, 14, 30 days
- minimum lag before conversion: 0h, 6h, 24h

> Side note: If channel credit changes a lot under small window/lag changes, the attribution policy may be fragile.


In [ ]:
def build_join_table(exposures_df: pd.DataFrame, conversions_df: pd.DataFrame, lookback_days: int, min_lag_hours: float = 0.0) -> pd.DataFrame:
    merged = conversions_df.merge(exposures_df, on='user_id', how='left')
    delta_hours = (merged['ts_conversion'] - merged['ts_exposure']).dt.total_seconds() / 3600
    valid = merged[(delta_hours >= min_lag_hours) & (delta_hours <= lookback_days * 24)].copy()
    if valid.empty:
        return pd.DataFrame(columns=[
            'lookback_days','min_lag_hours','conversion_id','user_id','ts_conversion','conversion_value',
            'exposure_id','channel','ts_exposure','hours_before_conversion'
        ])
    valid['hours_before_conversion'] = delta_hours.loc[valid.index]
    valid['lookback_days'] = lookback_days
    valid['min_lag_hours'] = min_lag_hours
    cols=['lookback_days','min_lag_hours','conversion_id','user_id','ts_conversion','conversion_value','exposure_id','channel','ts_exposure','hours_before_conversion']
    return valid[cols].sort_values(['conversion_id','ts_exposure']).reset_index(drop=True)

windows = [1, 3, 7, 14, 30]
lags = [0, 6, 24]
robust_rows = []
for w in windows:
    for lag in lags:
        j = build_join_table(exposures, conversions, lookback_days=w, min_lag_hours=lag)
        if j.empty:
            continue
        conv_cov = j['conversion_id'].nunique()
        total_conv = conversions['conversion_id'].nunique()
        for m in ATTRIBUTION_METHODS:
            out = run_method(j, m)
            total_credit = float(out['credit'].sum())
            top_channel = out.sort_values('credit', ascending=False)['channel'].iloc[0] if len(out) else None
            robust_rows.append({
                'window_days': w,
                'min_lag_hours': lag,
                'method': m,
                'total_credit': total_credit,
                'coverage_pct': round(100 * conv_cov / max(total_conv, 1), 2),
                'top_channel': top_channel
            })

robustness = pd.DataFrame(robust_rows)
display(robustness.head(20))

stability = robustness.groupby('method').agg(
    avg_coverage_pct=('coverage_pct', 'mean'),
    top_channel_nunique=('top_channel', 'nunique'),
    total_credit_std=('total_credit', 'std')
).reset_index().sort_values(['top_channel_nunique', 'total_credit_std'])
display(stability)


## 7) Segment-Level Sensitivity

Segments used:
- user touch-count cohort (low / medium / high)
- dominant channel cohort
- campaign cohort (bucketed campaign IDs)

> Side note: Segment sensitivity tells us whether conclusions hold across different user/channel contexts.


In [ ]:
# Build user-level segment labels from real data
user_touch = exposures.groupby('user_id').size().rename('touch_count').reset_index()
# Robust cohorting without qcut edge-duplication failures
touch_pct = user_touch['touch_count'].rank(method='average', pct=True)
user_touch['touch_cohort'] = pd.cut(touch_pct, bins=[0.0, 1/3, 2/3, 1.0], labels=['low', 'medium', 'high'], include_lowest=True)

dom = exposures.groupby(['user_id', 'channel']).size().rename('n').reset_index()
dom = dom.sort_values(['user_id', 'n'], ascending=[True, False]).drop_duplicates('user_id')
dom = dom[['user_id', 'channel']].rename(columns={'channel': 'dominant_channel'})

camp_tmp = exposures.copy()
camp_tmp['campaign_cohort'] = camp_tmp['channel'].astype(str).str[-1]  # real-data-based lightweight cohort key
camp_mode = camp_tmp.groupby(['user_id', 'campaign_cohort']).size().rename('n').reset_index()
camp_mode = camp_mode.sort_values(['user_id', 'n'], ascending=[True, False]).drop_duplicates('user_id')
camp_mode = camp_mode[['user_id', 'campaign_cohort']]

user_segments = user_touch.merge(dom, on='user_id', how='left').merge(camp_mode, on='user_id', how='left')
display(user_segments.head(10))

joined_seg = joined_7d.merge(user_segments, on='user_id', how='left')

def run_segment_slice(df: pd.DataFrame, segment_col: str) -> pd.DataFrame:
    rows = []
    for seg_val, g in df.groupby(segment_col, dropna=False):
        if g.empty:
            continue
        for m in ATTRIBUTION_METHODS:
            out = run_method(g, m)
            out['segment_col'] = segment_col
            out['segment_value'] = str(seg_val)
            rows.append(out)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['method', 'channel', 'credit', 'segment_col', 'segment_value'])

seg_results = pd.concat([
    run_segment_slice(joined_seg, 'touch_cohort'),
    run_segment_slice(joined_seg, 'dominant_channel'),
    run_segment_slice(joined_seg, 'campaign_cohort')
], ignore_index=True)

seg_summary = seg_results.groupby(['segment_col', 'segment_value', 'method'], as_index=False)['credit'].sum()
display(seg_summary.head(30))

# Top channel by segment/method
ranked = seg_results.sort_values(['segment_col', 'segment_value', 'method', 'credit'], ascending=[True, True, True, False])
top_by_segment = ranked.groupby(['segment_col', 'segment_value', 'method'], as_index=False).first()[['segment_col', 'segment_value', 'method', 'channel', 'credit']]
display(top_by_segment.head(30))



## 8) Measurement-Ready Output Table

This table is the core exposure->conversion dataset that downstream dashboards/modeling can consume.

> Side note: In production this is often persisted in a warehouse/Delta table with versioned logic.


In [ ]:
measurement_table = build_join_table(exposures, conversions, lookback_days=7, min_lag_hours=0)
print('Rows:', len(measurement_table))
print('Unique conversions:', measurement_table['conversion_id'].nunique())
display(measurement_table.head(15))


## 9) Key Learning

- Real-world exposure->conversion joining is now explicit and auditable.
- Attribution method choice, window choice, and lag assumptions all materially affect channel credit.
- Segment-level sensitivity highlights where a method is stable or unstable across user/channel contexts.
- This notebook now covers baseline and next-level attribution depth in one reproducible workflow.
